In [0]:
import uuid
from datetime import datetime, timezone

from pyspark.sql.types import (
    StructType, StructField, StringType,
    TimestampType, LongType
)

from pyspark.sql import functions as F

# Audit configuration
PIPELINE_NAME = "retail_data_pipeline"
TASK_NAME = "02_bronze_ingestion"
AUDIT_TABLE = "workspace.retail_training.pipeline_audit_log"

RUN_ID = str(uuid.uuid4())
print(f"Task audit ID: {RUN_ID}")


def write_audit_record(
    run_status,
    start_time,
    end_time,
    rows_processed=None,
    error_message=None
):
    audit_schema = StructType([
        StructField("pipeline_name", StringType(), False),
        StructField("task_name", StringType(), False),
        StructField("run_id", StringType(), True),
        StructField("run_status", StringType(), False),
        StructField("start_time", TimestampType(), True),
        StructField("end_time", TimestampType(), True),
        StructField("rows_processed", LongType(), True),
        StructField("error_message", StringType(), True),
        StructField("logged_at", TimestampType(), True)
    ])

    audit_record = [(
        PIPELINE_NAME,
        TASK_NAME,
        RUN_ID,
        run_status,
        start_time,
        end_time,
        rows_processed,
        error_message[:4000] if error_message else None,
        end_time
    )]

    audit_df = spark.createDataFrame(
        audit_record,
        schema=audit_schema
    )

    audit_df.write.format("delta").mode("append").saveAsTable(
        AUDIT_TABLE
    )



# Project configuration
CATALOG = "workspace"
SCHEMA = "retail_training"
RAW_PATH = "/Volumes/workspace/retail_training/raw_files"

start_time = datetime.now(timezone.utc)
rows_processed = 0

try:
    # Load the three dimension files
    dimension_files = ["customers", "products", "stores"]

    for name in dimension_files:
        df = (
            spark.read
            .option("header", "true")
            .option("inferSchema", "true")
            .csv(f"{RAW_PATH}/{name}.csv")
            .withColumn("_ingested_at", F.current_timestamp())
        )

        (
            df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(f"{CATALOG}.{SCHEMA}.bronze_{name}")
        )

        row_count = df.count()
        rows_processed += row_count
        print(f"Loaded bronze_{name}: {row_count} rows")

    # Load initial sales only if Bronze sales doesn't exist.
    sales_table = f"{CATALOG}.{SCHEMA}.bronze_sales"

    if not spark.catalog.tableExists(sales_table):
        initial_sales = (
            spark.read
            .option("header", "true")
            .option("inferSchema", "true")
            .csv(f"{RAW_PATH}/sales.csv")
            .withColumn("_ingested_at", F.current_timestamp())
        )

        (
            initial_sales.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(sales_table)
        )

        sales_count = initial_sales.count()
        rows_processed += sales_count
        print(f"Initial sales loaded: {sales_count} rows")
    else:
        print("bronze_sales already exists; initial file was not reloaded.")

    print("Bronze ingestion notebook completed.")

except Exception as e:
    end_time = datetime.now(timezone.utc)

    try:
        write_audit_record(
            run_status="FAILED",
            start_time=start_time,
            end_time=end_time,
            rows_processed=rows_processed,
            error_message=str(e)
        )
    except Exception as audit_error:
        print(f"WARNING: Failed to write failure audit: {audit_error}")

    print(f"BRONZE INGESTION FAILED: {e}")
    raise

else:
    end_time = datetime.now(timezone.utc)

    write_audit_record(
        run_status="SUCCESS",
        start_time=start_time,
        end_time=end_time,
        rows_processed=rows_processed
    )

    print(f"SUCCESS: Audited {rows_processed} rows")


Task audit ID: e1f543b7-54a7-408c-af58-6b62c00ad518
Loaded bronze_customers: 20 rows
Loaded bronze_products: 12 rows
Loaded bronze_stores: 5 rows
bronze_sales already exists; initial file was not reloaded.
Bronze ingestion notebook completed.
SUCCESS: Audited 37 rows


In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT sale_id) AS unique_sale_ids
FROM workspace.retail_training.bronze_sales;

total_rows,unique_sale_ids
105,105
